# Floquet Ising edge mode coupled to a dissipative TLS

## Numerical validation notebook, version 2

This notebook implements one model consistently from beginning to end. It replaces the two exploratory notebooks and is organized around the predictions of the theory note:

1. analytic and numerical bulk Floquet spectrum;
2. chiral invariants and open-boundary zero/pi Majorana modes;
3. finite-size edge-mode splitting;
4. TLS frequency response and phase;
5. dissipation crossover;
6. the one-period quantum-channel eigenvalue near minus one.
7. production-size frequency data, matched-window controls, and time-window diagnostics.

The executed copy uses a small fast configuration. Cells that may take more than five minutes at publication size are present but disabled by default through RUN_PRODUCTION = False.


In [ ]:
import os
os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib-floquet-tls")

from dataclasses import dataclass, replace
from pathlib import Path
import platform
import sys
import time

import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import numpy as np
from scipy.linalg import eig, expm
from scipy.optimize import curve_fit
from scipy.signal import find_peaks
from scipy.sparse import csr_matrix, eye, kron
from scipy.sparse.linalg import expm_multiply

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "font.size": 10,
})

FAST_MODE = True
RUN_PRODUCTION = False
RUN_MATCHED_N4_CONTROL = True
RUN_REPRESENTATIVE_N6 = True
CELL_TIMEOUT_POLICY_SECONDS = 300

print({
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "numpy": np.__version__,
    "fast_mode": FAST_MODE,
    "run_production": RUN_PRODUCTION,
    "run_matched_N4_control": RUN_MATCHED_N4_CONTROL,
    "run_representative_N6": RUN_REPRESENTATIVE_N6,
    "per_cell_runtime_policy_s": CELL_TIMEOUT_POLICY_SECONDS,
})


## 1. Unified conventions

The chain occupies sites j = 0,...,N-1 and the TLS occupies site d = N. The two-step drive is

\[
H_1=H_{ZZ}+H_d+H_{ed},\qquad
H_2=H_X+H_d+H_{ed},
\]

\[
H_{ZZ}=-J\sum_{j=0}^{N-2}Z_jZ_{j+1},\qquad
H_X=-h\sum_{j=0}^{N-1}X_j.
\]

The physical TLS ground state is the computational state \(|0\rangle\). Therefore

\[
H_d=-\frac{\omega_d}{2}Z_d,\qquad
\tau_- = |0\rangle\langle 1|.
\]

The transverse exchange coupling is kept on during both drive steps,

\[
H_{ed}=g(s_+^{0}\tau_-+s_-^{0}\tau_+)
=\frac{g}{2}(X_0X_d+Y_0Y_d).
\]

Dissipation acts only on the TLS:

\[
\dot\rho=-i[H_s,\rho]+\gamma_1\mathcal D[\tau_-]\rho
+\frac{\gamma_\phi}{2}\mathcal D[Z_d]\rho.
\]

All density matrices are vectorized in column-major order.


In [ ]:
I2 = csr_matrix(np.eye(2, dtype=complex))
X2 = csr_matrix(np.array([[0, 1], [1, 0]], dtype=complex))
Y2 = csr_matrix(np.array([[0, -1j], [1j, 0]], dtype=complex))
Z2 = csr_matrix(np.diag([1.0, -1.0]).astype(complex))
SM2 = csr_matrix(np.array([[0, 1], [0, 0]], dtype=complex))  # |0><1|
SP2 = SM2.getH()


def kron_all(factors):
    out = csr_matrix([[1.0 + 0.0j]])
    for factor in factors:
        out = kron(out, factor, format="csr")
    return out


def site_operator(local_operator, site, n_total):
    return kron_all([
        local_operator if j == site else I2
        for j in range(n_total)
    ])


def operator_lists(n_total):
    return {
        "x": [site_operator(X2, j, n_total) for j in range(n_total)],
        "y": [site_operator(Y2, j, n_total) for j in range(n_total)],
        "z": [site_operator(Z2, j, n_total) for j in range(n_total)],
        "sm": [site_operator(SM2, j, n_total) for j in range(n_total)],
    }


def zero_operator(dimension):
    return csr_matrix((dimension, dimension), dtype=complex)


def computational_ket(bits):
    index = 0
    for bit in bits:
        index = 2 * index + int(bit)
    ket = np.zeros(2 ** len(bits), dtype=complex)
    ket[index] = 1.0
    return ket


def density_vector_from_bits(bits):
    ket = computational_ket(bits)
    rho = np.outer(ket, ket.conjugate())
    return rho.reshape(-1, order="F")


def unvec(vector, dimension):
    return np.asarray(vector).reshape((dimension, dimension), order="F")


def expectation_from_vec(operator, vector, dimension):
    rho = unvec(vector, dimension)
    return float(np.trace(operator.toarray() @ rho).real)


def liouvillian(hamiltonian, collapse_operators):
    dimension = hamiltonian.shape[0]
    identity = eye(dimension, format="csr", dtype=complex)
    generator = -1j * (
        kron(identity, hamiltonian, format="csr")
        - kron(hamiltonian.T, identity, format="csr")
    )
    for collapse in collapse_operators:
        cdc = collapse.getH() @ collapse
        generator = generator + kron(collapse.conjugate(), collapse, format="csr")
        generator = generator - 0.5 * kron(identity, cdc, format="csr")
        generator = generator - 0.5 * kron(cdc.T, identity, format="csr")
    return generator.tocsr()


print("Local-operator algebra checks:")
print("  ||X^2-I|| =", np.linalg.norm((X2 @ X2 - I2).toarray()))
print("  ||[X,Y]-2iZ|| =", np.linalg.norm((X2 @ Y2 - Y2 @ X2 - 2j * Z2).toarray()))
print("  lowering matrix =", SM2.toarray().tolist())


In [ ]:
@dataclass(frozen=True)
class Parameters:
    N: int = 4
    J: float = 1.0
    h: float = 1.0
    alpha_over_pi: float = 0.75
    beta_over_pi: float = 0.90
    g: float = 0.08
    omega_d: float | None = None
    gamma1: float = 0.08
    gamma_phi: float = 0.0
    periods: int = 32
    samples_per_step: int = 2

    @property
    def alpha(self):
        return self.alpha_over_pi * np.pi

    @property
    def beta(self):
        return self.beta_over_pi * np.pi

    @property
    def T1(self):
        return self.beta / (2.0 * self.J)

    @property
    def T2(self):
        return self.alpha / (2.0 * self.h)

    @property
    def T(self):
        return self.T1 + self.T2

    @property
    def Omega(self):
        return 2.0 * np.pi / self.T

    @property
    def tls_frequency(self):
        return self.Omega / 2.0 if self.omega_d is None else self.omega_d


p_fast = Parameters()
print(p_fast)
print({
    "T1": p_fast.T1,
    "T2": p_fast.T2,
    "T": p_fast.T,
    "Omega": p_fast.Omega,
    "Omega_over_2": p_fast.Omega / 2.0,
    "hT2_over_pi": p_fast.h * p_fast.T2 / np.pi,
    "JT1_over_pi": p_fast.J * p_fast.T1 / np.pi,
})


## 2. Closed-chain bulk spectrum

With

\[
\alpha=2hT_2,\qquad \beta=2JT_1,
\]

the exact bulk dispersion is

\[
\cos(\varepsilon_kT)=
\cos\alpha\cos\beta+\cos k\,\sin\alpha\sin\beta.
\]

The first numerical check compares this expression with the trace of the explicit two-by-two Majorana rotation matrix.


In [ ]:
sigma_x = np.array([[0, 1], [1, 0]], dtype=complex)
sigma_y = np.array([[0, -1j], [1j, 0]], dtype=complex)
identity_2 = np.eye(2, dtype=complex)


def su2_rotation(theta, nx, ny):
    return (
        np.cos(theta) * identity_2
        - 1j * np.sin(theta) * (nx * sigma_x + ny * sigma_y)
    )


def bulk_floquet_matrix(k, alpha, beta):
    r_x = su2_rotation(alpha, 0.0, 1.0)
    r_zz = su2_rotation(beta, np.sin(k), -np.cos(k))
    return r_x @ r_zz


k_values = np.linspace(-np.pi, np.pi, 801)
cos_eps_analytic = (
    np.cos(p_fast.alpha) * np.cos(p_fast.beta)
    + np.cos(k_values) * np.sin(p_fast.alpha) * np.sin(p_fast.beta)
)
cos_eps_numeric = np.array([
    0.5 * np.trace(bulk_floquet_matrix(k, p_fast.alpha, p_fast.beta)).real
    for k in k_values
])
epsilon = np.arccos(np.clip(cos_eps_analytic, -1.0, 1.0)) / p_fast.T

zero_gap = float(np.min(np.arccos(np.clip(cos_eps_analytic, -1.0, 1.0))) / p_fast.T)
pi_gap = float(np.min(np.pi - np.arccos(np.clip(cos_eps_analytic, -1.0, 1.0))) / p_fast.T)

print("max analytic/numeric cosine error =", np.max(np.abs(cos_eps_analytic - cos_eps_numeric)))
print("bulk zero gap =", zero_gap)
print("bulk pi gap =", pi_gap)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(k_values / np.pi, epsilon, label=r"$+\varepsilon_k$")
ax.plot(k_values / np.pi, -epsilon, label=r"$-\varepsilon_k$")
ax.axhline(np.pi / p_fast.T, color="k", ls=":", lw=1, label=r"$\pi/T$")
ax.set(xlabel=r"$k/\pi$", ylabel="quasienergy", title="Analytic bulk Floquet spectrum")
ax.legend(ncol=3)
plt.show()


## 3. Chiral time frames and the two invariants

The two symmetric frames are evaluated directly. Their off-diagonal windings are combined as

\[
\nu_0=\frac{\nu'+\nu''}{2},\qquad
\nu_\pi=\frac{\nu'-\nu''}{2}.
\]

The selected working point is required to lie in the coexistence region with nonzero zero- and pi-gap invariants.


In [ ]:
def complex_winding(values):
    values = np.asarray(values, dtype=complex)
    phase_steps = np.angle(np.roll(values, -1) * np.conjugate(values))
    return int(np.rint(np.sum(phase_steps) / (2.0 * np.pi)))


def chiral_invariants(alpha, beta, nk=501, singular_tolerance=1e-7):
    ks = np.linspace(-np.pi, np.pi, nk, endpoint=False)
    ux = su2_rotation(alpha, 0.0, 1.0)
    ux_half = su2_rotation(alpha / 2.0, 0.0, 1.0)
    offdiag_prime = []
    offdiag_double_prime = []
    for k in ks:
        uzz = su2_rotation(beta, np.sin(k), -np.cos(k))
        uzz_half = su2_rotation(beta / 2.0, np.sin(k), -np.cos(k))
        u_prime = ux_half @ uzz @ ux_half
        u_double_prime = uzz_half @ ux @ uzz_half
        offdiag_prime.append(u_prime[0, 1])
        offdiag_double_prime.append(u_double_prime[0, 1])
    offdiag_prime = np.asarray(offdiag_prime)
    offdiag_double_prime = np.asarray(offdiag_double_prime)
    if min(np.min(np.abs(offdiag_prime)), np.min(np.abs(offdiag_double_prime))) < singular_tolerance:
        return np.nan, np.nan
    winding_prime = complex_winding(offdiag_prime)
    winding_double_prime = complex_winding(offdiag_double_prime)
    return (
        int(np.rint((winding_prime + winding_double_prime) / 2.0)),
        int(np.rint((winding_prime - winding_double_prime) / 2.0)),
    )


nu0_selected, nupi_selected = chiral_invariants(p_fast.alpha, p_fast.beta)
print("selected point (nu0, nupi) =", (nu0_selected, nupi_selected))
assert abs(nu0_selected) == 1 and abs(nupi_selected) == 1

grid_size = 33 if FAST_MODE else 81
alpha_grid = np.linspace(0.05, 0.95, grid_size) * np.pi
beta_grid = np.linspace(0.05, 0.95, grid_size) * np.pi
phase_code = np.full((grid_size, grid_size), np.nan)

t0 = time.perf_counter()
for ib, beta in enumerate(beta_grid):
    for ia, alpha in enumerate(alpha_grid):
        nu0, nupi = chiral_invariants(alpha, beta, nk=241 if FAST_MODE else 801)
        if np.isfinite(nu0) and np.isfinite(nupi):
            phase_code[ib, ia] = 2 * int(abs(nu0) > 0) + int(abs(nupi) > 0)
print("invariant map runtime (s) =", time.perf_counter() - t0)

cmap = ListedColormap(["#dddddd", "#3b82f6", "#ef4444", "#8b5cf6"])
fig, ax = plt.subplots(figsize=(6.2, 5.2))
image = ax.imshow(
    phase_code,
    origin="lower",
    extent=[alpha_grid[0] / np.pi, alpha_grid[-1] / np.pi,
            beta_grid[0] / np.pi, beta_grid[-1] / np.pi],
    aspect="auto",
    interpolation="nearest",
    vmin=-0.5,
    vmax=3.5,
    cmap=cmap,
)
ax.plot(p_fast.alpha_over_pi, p_fast.beta_over_pi, "ko", ms=6, label="working point")
ax.set(
    xlabel=r"$\alpha/\pi$",
    ylabel=r"$\beta/\pi$",
    title=r"Floquet phases: gray trivial, blue $\pi$, red 0, purple coexistence",
)
ax.legend(loc="lower right")
plt.show()


## 4. Open-boundary Majorana modes

The real orthogonal one-cycle Majorana rotation is diagonalized under open boundary conditions. Degenerate conjugate eigenvectors are recombined by diagonalizing the position operator inside the near-zero or near-pi subspace, producing a left-localized representative.


In [ ]:
def majorana_generators(N, J=1.0, h=1.0):
    a_zz = np.zeros((2 * N, 2 * N), dtype=float)
    a_x = np.zeros((2 * N, 2 * N), dtype=float)
    for j in range(N):
        a_x[2 * j, 2 * j + 1] = -2.0 * h
        a_x[2 * j + 1, 2 * j] = 2.0 * h
    for j in range(N - 1):
        a_zz[2 * j + 1, 2 * j + 2] = -2.0 * J
        a_zz[2 * j + 2, 2 * j + 1] = 2.0 * J
    return a_zz, a_x


def majorana_floquet(N, parameters):
    a_zz, a_x = majorana_generators(N, parameters.J, parameters.h)
    return expm(a_x * parameters.T2) @ expm(a_zz * parameters.T1)


def special_mode_data(rotation, target_phase, subspace_dimension=2):
    eigenvalues, eigenvectors = eig(rotation)
    phases = np.angle(eigenvalues)
    distance = np.abs(np.angle(np.exp(1j * (phases - target_phase))))
    indices = np.argsort(distance)[:subspace_dimension]
    subspace, _ = np.linalg.qr(eigenvectors[:, indices])
    N = rotation.shape[0] // 2
    position = np.repeat(np.arange(N, dtype=float), 2)
    projected_position = subspace.conjugate().T @ (position[:, None] * subspace)
    _, local_vectors = np.linalg.eigh(projected_position)
    left_mode = subspace @ local_vectors[:, 0]
    weights = np.array([
        np.sum(np.abs(left_mode[2 * j:2 * j + 2]) ** 2)
        for j in range(N)
    ])
    weights = weights / np.sum(weights)
    return {
        "distance": float(distance[indices[0]]),
        "weights": weights,
        "phases": phases,
        "eigenvectors": eigenvectors,
        "eigenvalues": eigenvalues,
    }


N_majorana = 40
rotation = majorana_floquet(N_majorana, p_fast)
zero_mode = special_mode_data(rotation, 0.0)
pi_mode = special_mode_data(rotation, np.pi)
orthogonality_error = np.linalg.norm(rotation.T @ rotation - np.eye(2 * N_majorana))

eigenvalues, eigenvectors = eig(rotation)
phases = np.angle(eigenvalues)
edge_weights = np.sum(np.abs(eigenvectors[:4, :]) ** 2, axis=0)
edge_weights += np.sum(np.abs(eigenvectors[-4:, :]) ** 2, axis=0)

print("orthogonality error =", orthogonality_error)
print("zero-mode phase distance =", zero_mode["distance"])
print("pi-mode phase distance =", pi_mode["distance"])

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
scatter = axes[0].scatter(np.arange(len(phases)), phases, c=edge_weights, cmap="viridis", s=18)
axes[0].axhline(0, color="k", ls=":", lw=1)
axes[0].axhline(np.pi, color="k", ls=":", lw=1)
axes[0].axhline(-np.pi, color="k", ls=":", lw=1)
axes[0].set(xlabel="single-particle mode index", ylabel="Floquet phase",
            title="Open-chain Floquet spectrum")
fig.colorbar(scatter, ax=axes[0], label="edge weight")

sites = np.arange(N_majorana)
axes[1].semilogy(sites, zero_mode["weights"] + 1e-18, "o-", ms=3, label="zero mode")
axes[1].semilogy(sites, pi_mode["weights"] + 1e-18, "s-", ms=3, label="pi mode")
axes[1].set(xlabel="site", ylabel="Majorana weight", title="Left-localized mode profiles")
axes[1].legend()
fig.tight_layout()
plt.show()


## 5. Closed-chain finite-size splitting

For each chain length, the circular eigenphase distances to zero and pi are extracted. An exponential fit estimates the localization lengths independently of any time-domain window. Fits use only splittings safely above the double-precision floor; unresolved points are displayed separately and are never used as evidence for a continued exponential law.


In [ ]:
N_values = np.arange(2, 33, 1) if FAST_MODE else np.arange(2, 81, 1)
delta_zero = []
delta_pi = []

t0 = time.perf_counter()
for N in N_values:
    r_f = majorana_floquet(int(N), p_fast)
    phases = np.angle(np.linalg.eigvals(r_f))
    delta_zero.append(np.min(np.abs(np.angle(np.exp(1j * phases)))))
    delta_pi.append(np.min(np.abs(np.angle(-np.exp(1j * phases)))))

delta_zero = np.asarray(delta_zero)
delta_pi = np.asarray(delta_pi)


def exponential_fit_size(sizes, splittings, lower=1e-13):
    mask = (splittings > lower) & np.isfinite(splittings)
    if np.count_nonzero(mask) < 4:
        raise RuntimeError("Too few numerically resolved points for an exponential fit.")
    slope, intercept = np.polyfit(sizes[mask], np.log(splittings[mask]), 1)
    return -1.0 / slope, np.exp(intercept), mask


xi_zero, prefactor_zero, mask_zero = exponential_fit_size(N_values, delta_zero)
xi_pi, prefactor_pi, mask_pi = exponential_fit_size(N_values, delta_pi)
print("finite-size runtime (s) =", time.perf_counter() - t0)
print("xi_zero =", xi_zero, "xi_pi =", xi_pi)

fig, ax = plt.subplots(figsize=(7, 4.3))
ax.semilogy(N_values[mask_zero], delta_zero[mask_zero], "o", label="zero mode: resolved")
ax.semilogy(N_values[mask_pi], delta_pi[mask_pi], "s", label="pi mode: resolved")
ax.semilogy(
    N_values[~mask_zero], np.maximum(delta_zero[~mask_zero], 1e-16),
    "o", mfc="none", color="C0", alpha=0.5, label="zero mode: precision limited",
)
ax.semilogy(
    N_values[~mask_pi], np.maximum(delta_pi[~mask_pi], 1e-16),
    "s", mfc="none", color="C1", alpha=0.5, label="pi mode: precision limited",
)
fit_sizes_zero = N_values[mask_zero]
fit_sizes_pi = N_values[mask_pi]
ax.semilogy(fit_sizes_zero, prefactor_zero * np.exp(-fit_sizes_zero / xi_zero), "-", color="C0")
ax.semilogy(fit_sizes_pi, prefactor_pi * np.exp(-fit_sizes_pi / xi_pi), "-", color="C1")
ax.axhline(1e-13, color="0.4", ls=":", lw=1, label="fit cutoff")
ax.set(xlabel="chain length N", ylabel="circular eigenphase distance",
       title="Closed-chain finite-size splitting")
ax.legend()
plt.show()


## 6. Closed many-body edge response

This calculation is separate from the Majorana eigenmode calculation. It verifies that a simple product state has a strong boundary period-doubled component while the selected bulk spin has a much weaker long-time component at the working point.


In [ ]:
def build_closed_chain(N, parameters):
    ops = operator_lists(N)
    dimension = 2 ** N
    h_zz = zero_operator(dimension)
    h_x = zero_operator(dimension)
    for j in range(N - 1):
        h_zz = h_zz - parameters.J * (ops["z"][j] @ ops["z"][j + 1])
    for j in range(N):
        h_x = h_x - parameters.h * ops["x"][j]
    return ops, h_zz.tocsr(), h_x.tocsr()


def signed_pi_component(signal, discard_fraction=0.25):
    signal = np.asarray(signal, dtype=float)
    start = int(np.floor(discard_fraction * len(signal)))
    y = signal[start:]
    return float(np.mean(((-1.0) ** np.arange(len(y))) * y))


N_spin = 6
ops_spin, h_zz_spin, h_x_spin = build_closed_chain(N_spin, p_fast)
psi = computational_ket([0] * N_spin)
edge_trace = []
bulk_trace = []

for n in range(60):
    edge_trace.append(float(np.vdot(psi, ops_spin["z"][0] @ psi).real))
    bulk_trace.append(float(np.vdot(psi, ops_spin["z"][N_spin // 2] @ psi).real))
    psi = expm_multiply(-1j * h_zz_spin * p_fast.T1, psi)
    psi = expm_multiply(-1j * h_x_spin * p_fast.T2, psi)

print("edge |M_pi| =", abs(signed_pi_component(edge_trace)))
print("bulk |M_pi| =", abs(signed_pi_component(bulk_trace)))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(edge_trace, "o-", ms=3, label="left edge")
ax.plot(bulk_trace, "s--", ms=3, label="bulk")
ax.set(xlabel="Floquet period n", ylabel=r"$\langle Z_j\rangle$",
       title="Closed-chain stroboscopic response")
ax.legend()
plt.show()


## 7. Open chain plus TLS

The following construction adds the defect Hamiltonian and TLS-only collapse operators to the same chain parameters. It also verifies the exchange identity and Hermiticity before any time evolution.


In [ ]:
def build_open_model(parameters):
    N = parameters.N
    n_total = N + 1
    d_site = N
    ops = operator_lists(n_total)
    dimension = 2 ** n_total
    h_zz = zero_operator(dimension)
    h_x = zero_operator(dimension)
    for j in range(N - 1):
        h_zz = h_zz - parameters.J * (ops["z"][j] @ ops["z"][j + 1])
    for j in range(N):
        h_x = h_x - parameters.h * ops["x"][j]

    h_d = -0.5 * parameters.tls_frequency * ops["z"][d_site]
    h_ed = parameters.g * (
        ops["sm"][0].getH() @ ops["sm"][d_site]
        + ops["sm"][0] @ ops["sm"][d_site].getH()
    )
    h_xy = 0.5 * parameters.g * (
        ops["x"][0] @ ops["x"][d_site]
        + ops["y"][0] @ ops["y"][d_site]
    )

    collapse_operators = []
    if parameters.gamma1 > 0:
        collapse_operators.append(np.sqrt(parameters.gamma1) * ops["sm"][d_site])
    if parameters.gamma_phi > 0:
        collapse_operators.append(np.sqrt(parameters.gamma_phi / 2.0) * ops["z"][d_site])

    h1 = (h_zz + h_d + h_ed).tocsr()
    h2 = (h_x + h_d + h_ed).tocsr()
    identity = eye(dimension, format="csr", dtype=complex)
    tau_z = -ops["z"][d_site]
    p_excited = 0.5 * (identity + tau_z)

    checks = {
        "exchange_identity_error": np.linalg.norm((h_ed - h_xy).toarray()),
        "H1_hermiticity_error": np.linalg.norm((h1 - h1.getH()).toarray()),
        "H2_hermiticity_error": np.linalg.norm((h2 - h2.getH()).toarray()),
        "collapse_sites": ["TLS"] * len(collapse_operators),
    }
    observables = {
        "edge": ops["z"][0],
        "bulk": ops["z"][N // 2],
        "tls_z": tau_z.tocsr(),
        "tls_x": ops["x"][d_site],
        "tls_y": ops["y"][d_site],
        "tls_excited": p_excited.tocsr(),
    }
    return {
        "dimension": dimension,
        "ops": ops,
        "H1": h1,
        "H2": h2,
        "collapse": collapse_operators,
        "observables": observables,
        "checks": checks,
    }


model_check = build_open_model(p_fast)
print(model_check["checks"])
assert model_check["checks"]["exchange_identity_error"] < 1e-12
assert model_check["checks"]["H1_hermiticity_error"] < 1e-12
assert model_check["checks"]["H2_hermiticity_error"] < 1e-12


In [ ]:
def simulate_open(parameters, validate=True):
    model = build_open_model(parameters)
    dimension = model["dimension"]
    l1 = liouvillian(model["H1"], model["collapse"])
    l2 = liouvillian(model["H2"], model["collapse"])
    vector = density_vector_from_bits([0] * parameters.N + [0])
    observables = model["observables"]

    continuous_time = []
    continuous = {name: [] for name in observables}
    stroboscopic = {name: [] for name in observables}
    trace_errors = []
    hermiticity_errors = []
    minimum_eigenvalues = []

    current_time = 0.0
    for period in range(parameters.periods):
        for name, operator in observables.items():
            stroboscopic[name].append(expectation_from_vec(operator, vector, dimension))

        if validate:
            rho = unvec(vector, dimension)
            trace_errors.append(abs(np.trace(rho) - 1.0))
            hermiticity_errors.append(np.linalg.norm(rho - rho.conjugate().T))
            minimum_eigenvalues.append(float(np.min(np.linalg.eigvalsh(rho)).real))

        trajectory_1 = expm_multiply(
            l1,
            vector,
            start=0.0,
            stop=parameters.T1,
            num=parameters.samples_per_step + 1,
            endpoint=True,
        )
        for sample in range(parameters.samples_per_step):
            continuous_time.append(current_time + sample * parameters.T1 / parameters.samples_per_step)
            for name, operator in observables.items():
                continuous[name].append(
                    expectation_from_vec(operator, trajectory_1[sample], dimension)
                )
        vector = trajectory_1[-1]
        current_time += parameters.T1

        trajectory_2 = expm_multiply(
            l2,
            vector,
            start=0.0,
            stop=parameters.T2,
            num=parameters.samples_per_step + 1,
            endpoint=True,
        )
        for sample in range(parameters.samples_per_step):
            continuous_time.append(current_time + sample * parameters.T2 / parameters.samples_per_step)
            for name, operator in observables.items():
                continuous[name].append(
                    expectation_from_vec(operator, trajectory_2[sample], dimension)
                )
        vector = trajectory_2[-1]
        current_time += parameters.T2

    for name, operator in observables.items():
        stroboscopic[name].append(expectation_from_vec(operator, vector, dimension))

    if validate:
        rho = unvec(vector, dimension)
        trace_errors.append(abs(np.trace(rho) - 1.0))
        hermiticity_errors.append(np.linalg.norm(rho - rho.conjugate().T))
        minimum_eigenvalues.append(float(np.min(np.linalg.eigvalsh(rho)).real))

    return {
        "parameters": parameters,
        "model": model,
        "time": np.asarray(continuous_time),
        "continuous": {key: np.asarray(value) for key, value in continuous.items()},
        "stroboscopic": {key: np.asarray(value) for key, value in stroboscopic.items()},
        "quality": {
            "max_trace_error": float(np.max(trace_errors)) if trace_errors else np.nan,
            "max_hermiticity_error": float(np.max(hermiticity_errors)) if hermiticity_errors else np.nan,
            "min_eigenvalue": float(np.min(minimum_eigenvalues)) if minimum_eigenvalues else np.nan,
        },
        "final_vector": vector,
    }


def complex_subharmonic_phasor(time_array, signal, omega, discard_time):
    time_array = np.asarray(time_array)
    signal = np.asarray(signal)
    mask = time_array >= discard_time
    t = time_array[mask]
    x = signal[mask]
    if len(t) < 4:
        return np.nan + 1j * np.nan
    x = x - np.mean(x)
    duration = t[-1] - t[0]
    return 2.0 * np.trapezoid(x * np.exp(1j * omega * t / 2.0), t) / duration


def run_metrics(run, discard_periods=8):
    parameters = run["parameters"]
    discard_time = discard_periods * parameters.T
    z_edge = complex_subharmonic_phasor(
        run["time"], run["continuous"]["edge"], parameters.Omega, discard_time
    )
    tls_lowering = 0.5 * (
        run["continuous"]["tls_x"] + 1j * run["continuous"]["tls_y"]
    )
    z_tls = complex_subharmonic_phasor(
        run["time"], tls_lowering, parameters.Omega, discard_time
    )
    z_tls_population = complex_subharmonic_phasor(
        run["time"], run["continuous"]["tls_z"], parameters.Omega, discard_time
    )
    late_mask = run["time"] >= discard_time
    emission = parameters.gamma1 * np.mean(run["continuous"]["tls_excited"][late_mask])
    phase = np.angle(z_tls / z_edge) if abs(z_edge) > 1e-10 and abs(z_tls) > 1e-10 else np.nan
    return {
        "A_edge": float(abs(z_edge)),
        "A_tls_transverse": float(abs(z_tls)),
        "A_tls_population": float(abs(z_tls_population)),
        "phase_tls_minus_edge": float(phase),
        "tls_emission": float(emission),
        "Mpi_edge_strobe": abs(signed_pi_component(run["stroboscopic"]["edge"], 0.25)),
        "Mpi_tls_population_strobe": abs(
            signed_pi_component(run["stroboscopic"]["tls_z"], 0.25)
        ),
    }


## 8. Open-system sanity checks

The zero-coupling case is compared with the independently evolved closed chain. Trace, Hermiticity, and positivity are checked for the dissipative baseline.


In [ ]:
p_decoupled = replace(
    p_fast,
    g=0.0,
    gamma1=0.0,
    gamma_phi=0.0,
    omega_d=p_fast.Omega / 2.0,
    periods=24,
)
t0 = time.perf_counter()
run_decoupled = simulate_open(p_decoupled, validate=True)
print("decoupled runtime (s) =", time.perf_counter() - t0)
print("decoupled quality =", run_decoupled["quality"])

ops_reference, hzz_reference, hx_reference = build_closed_chain(p_decoupled.N, p_decoupled)
psi_reference = computational_ket([0] * p_decoupled.N)
edge_reference = []
for period in range(p_decoupled.periods + 1):
    edge_reference.append(float(np.vdot(
        psi_reference, ops_reference["z"][0] @ psi_reference
    ).real))
    if period < p_decoupled.periods:
        psi_reference = expm_multiply(-1j * hzz_reference * p_decoupled.T1, psi_reference)
        psi_reference = expm_multiply(-1j * hx_reference * p_decoupled.T2, psi_reference)

decoupled_difference = np.max(np.abs(
    np.asarray(edge_reference) - run_decoupled["stroboscopic"]["edge"]
))
print("closed/open-decoupled edge difference =", decoupled_difference)
assert decoupled_difference < 1e-8

p_baseline = replace(p_fast, omega_d=p_fast.Omega / 2.0)
t0 = time.perf_counter()
run_baseline = simulate_open(p_baseline, validate=True)
baseline_runtime = time.perf_counter() - t0
print("baseline runtime (s) =", baseline_runtime)
print("baseline quality =", run_baseline["quality"])
print("baseline metrics =", run_metrics(run_baseline))
print(
    "max TLS transverse one-point signal =",
    max(
        np.max(np.abs(run_baseline["continuous"]["tls_x"])),
        np.max(np.abs(run_baseline["continuous"]["tls_y"])),
    ),
)


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
axes[0].plot(
    np.arange(len(run_baseline["stroboscopic"]["edge"])),
    run_baseline["stroboscopic"]["edge"],
    "o-",
    ms=3,
    label="edge",
)
axes[0].plot(
    np.arange(len(run_baseline["stroboscopic"]["bulk"])),
    run_baseline["stroboscopic"]["bulk"],
    "s--",
    ms=3,
    label="bulk",
)
axes[0].plot(
    np.arange(len(run_baseline["stroboscopic"]["tls_z"])),
    run_baseline["stroboscopic"]["tls_z"],
    ".-",
    label="TLS physical z",
)
axes[0].set(ylabel="stroboscopic expectation", title="Open-system baseline")
axes[0].legend(ncol=3)

axes[1].plot(
    run_baseline["time"] / p_baseline.T,
    run_baseline["continuous"]["tls_excited"],
    label="TLS excited population",
)
axes[1].plot(
    run_baseline["time"] / p_baseline.T,
    run_baseline["continuous"]["edge"],
    alpha=0.7,
    label="edge continuous signal",
)
axes[1].plot(
    run_baseline["time"] / p_baseline.T,
    run_baseline["continuous"]["tls_x"],
    alpha=0.7,
    label=r"TLS $\langle\tau_x\rangle$",
)
axes[1].set(xlabel=r"time / $T$", ylabel="continuous-time observable")
axes[1].legend()
fig.tight_layout()
plt.show()


## 9. Fast TLS frequency sweep

The most direct resonance observables are:

- suppression or reshaping of the edge subharmonic amplitude;
- the resonant Fourier component of \(\langle\tau_-\rangle=(\langle\tau_x\rangle+i\langle\tau_y\rangle)/2\);
- the dissipated TLS power proxy \(\gamma_1\overline{p_e}\);
- the continuous-time relative phase.

The phase is not inferred from period-end samples because the subharmonic is the stroboscopic Nyquist frequency.

The complex TLS phasor is extracted from \(\langle\tau_-\rangle\), so its magnitude and phase are the numerical observables most directly comparable with the linear susceptibility \(c_d/c_e\). TLS population modulation and the emission proxy are kept as independent, symmetry-allowed loading diagnostics.


In [ ]:
frequency_points = 21 if FAST_MODE else 61
frequency_ratios = np.linspace(0.55, 1.45, frequency_points)
frequency_rows = []

t0 = time.perf_counter()
for ratio in frequency_ratios:
    p_frequency = replace(
        p_fast,
        omega_d=ratio * p_fast.Omega / 2.0,
        periods=32 if FAST_MODE else 80,
        samples_per_step=2 if FAST_MODE else 4,
    )
    run = simulate_open(p_frequency, validate=False)
    frequency_rows.append({
        "omega_ratio": ratio,
        "omega_d": p_frequency.tls_frequency,
        **run_metrics(run, discard_periods=8 if FAST_MODE else 20),
    })
frequency_runtime = time.perf_counter() - t0
print("frequency sweep runtime (s) =", frequency_runtime)

frequency_data = {
    key: np.array([row[key] for row in frequency_rows])
    for key in frequency_rows[0]
}


def lorentzian_with_offset(omega, offset, amplitude, center, width):
    return offset + amplitude * width ** 2 / ((omega - center) ** 2 + width ** 2)


try:
    initial_guess = [
        float(np.min(frequency_data["tls_emission"])),
        float(np.max(frequency_data["tls_emission"]) - np.min(frequency_data["tls_emission"])),
        p_fast.Omega / 2.0,
        max(p_fast.gamma1 / 2.0, 0.03),
    ]
    fit_parameters, _ = curve_fit(
        lorentzian_with_offset,
        frequency_data["omega_d"],
        frequency_data["tls_emission"],
        p0=initial_guess,
        bounds=(
            [0.0, 0.0, 0.5 * p_fast.Omega / 2.0, 1e-4],
            [np.inf, np.inf, 1.5 * p_fast.Omega / 2.0, p_fast.Omega],
        ),
        maxfev=20000,
    )
    fit_success = True
except Exception as error:
    print("Lorentzian fit skipped:", repr(error))
    fit_parameters = np.array([np.nan] * 4)
    fit_success = False

print("Lorentzian fit [offset, amplitude, center, width] =", fit_parameters)
if fit_success:
    print("fitted center / (Omega/2) =", fit_parameters[2] / (p_fast.Omega / 2.0))

fig, axes = plt.subplots(2, 2, figsize=(11, 7), sharex=True)
x = frequency_data["omega_ratio"]
axes[0, 0].plot(x, frequency_data["A_edge"], "o-", label="edge")
axes[0, 0].plot(x, frequency_data["A_tls_transverse"], "s-", label=r"TLS $\tau_-$")
axes[0, 0].set(ylabel=r"continuous $A_{\Omega/2}$", title="Edge and transverse TLS response")
axes[0, 0].legend()

axes[0, 1].plot(x, frequency_data["tls_emission"], "o", label=r"$\gamma_1\overline{p_e}$")
if fit_success:
    dense_omega = np.linspace(
        frequency_data["omega_d"].min(), frequency_data["omega_d"].max(), 500
    )
    axes[0, 1].plot(
        dense_omega / (p_fast.Omega / 2.0),
        lorentzian_with_offset(dense_omega, *fit_parameters),
        "-",
        label="Lorentzian guide",
    )
axes[0, 1].set(ylabel="TLS emission proxy", title="Frequency-selective dissipation")
axes[0, 1].legend()

axes[1, 0].plot(x, frequency_data["phase_tls_minus_edge"], "o-")
axes[1, 0].axhline(0, color="k", ls=":", lw=1)
axes[1, 0].set(xlabel=r"$\omega_d/(\Omega/2)$", ylabel="relative phase [rad]",
               title=r"Transverse TLS phase relative to edge")

axes[1, 1].plot(x, frequency_data["Mpi_edge_strobe"], "o-", label="edge")
axes[1, 1].plot(
    x, frequency_data["Mpi_tls_population_strobe"], "s-", label=r"TLS population axis"
)
axes[1, 1].set(xlabel=r"$\omega_d/(\Omega/2)$", ylabel=r"$|M_\pi|$",
               title="Signed stroboscopic components")
axes[1, 1].legend()
fig.tight_layout()
plt.show()


## 10. Dissipation crossover

The adiabatic formula applies only in the fast-defect regime. This sweep therefore reports the full dynamics rather than extending the large-gamma expression into the coherent regime. The emission proxy and the edge response are shown together.


In [ ]:
gamma_points = 21 if FAST_MODE else 51
gamma_values = np.geomspace(0.005, 3.0, gamma_points)
gamma_rows = []

t0 = time.perf_counter()
for gamma1 in gamma_values:
    p_gamma = replace(
        p_fast,
        omega_d=p_fast.Omega / 2.0,
        gamma1=float(gamma1),
        periods=32 if FAST_MODE else 80,
        samples_per_step=2 if FAST_MODE else 4,
    )
    run = simulate_open(p_gamma, validate=False)
    gamma_rows.append({"gamma1": gamma1, **run_metrics(run, discard_periods=8)})
gamma_runtime = time.perf_counter() - t0
print("gamma sweep runtime (s) =", gamma_runtime)

gamma_data = {
    key: np.array([row[key] for row in gamma_rows])
    for key in gamma_rows[0]
}
emission_peak_index = int(np.argmax(gamma_data["tls_emission"]))
print("maximum TLS emission at gamma1 =", gamma_data["gamma1"][emission_peak_index])

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].semilogx(gamma_data["gamma1"], gamma_data["A_edge"], "o-", label="edge")
axes[0].semilogx(
    gamma_data["gamma1"], gamma_data["A_tls_transverse"], "s-", label=r"TLS $\tau_-$"
)
axes[0].set(xlabel=r"$\gamma_1$", ylabel=r"continuous $A_{\Omega/2}$",
            title="Subharmonic response versus TLS damping")
axes[0].legend()

axes[1].loglog(
    gamma_data["gamma1"], gamma_data["tls_emission"], "o-",
    label=r"$\gamma_1\overline{p_e}$",
)
axes[1].axvline(
    gamma_data["gamma1"][emission_peak_index],
    color="k",
    ls=":",
    lw=1,
    label="maximum loading",
)
axes[1].set(xlabel=r"$\gamma_1$", ylabel="TLS emission proxy",
            title="Coherent-to-fast-defect crossover")
axes[1].legend()
fig.tight_layout()
plt.show()


## 11. Small-system Floquet-channel spectrum

For a short chain the complete one-cycle superoperator can be constructed explicitly:

\[
\mathcal F=e^{\mathcal L_2T_2}e^{\mathcal L_1T_1}.
\]

The eigen-decomposition is also projected onto the chosen initial state and the left-edge observable. Among eigenvalues in a pi-phase window, we select the mode with the largest visibility

\[
\left|\operatorname{Tr}(Z_0R_\mu)\,(L_\mu|\rho_0)\right|,
\]

rather than automatically selecting the longest-lived mode. This prevents a nearly isolated far-edge mode from being misidentified as the decay channel visible at the TLS-coupled boundary. The selected eigenvalue is reported together with

\[
\tau_\pi=-\frac{T}{\ln|\lambda_\pi|}.
\]

This is a structural smoke test. Publication-size channel spectra require a matrix-free iterative calculation and are left to the disabled production section.


In [ ]:
p_channel = replace(
    p_fast,
    N=3,
    periods=40,
    omega_d=p_fast.Omega / 2.0,
    samples_per_step=2,
)
model_channel = build_open_model(p_channel)
l1_channel = liouvillian(model_channel["H1"], model_channel["collapse"]).toarray()
l2_channel = liouvillian(model_channel["H2"], model_channel["collapse"]).toarray()

t0 = time.perf_counter()
channel = expm(l2_channel * p_channel.T2) @ expm(l1_channel * p_channel.T1)
channel_eigenvalues, channel_right_vectors = np.linalg.eig(channel)
channel_runtime = time.perf_counter() - t0

initial_vector = density_vector_from_bits([0] * p_channel.N + [0])
expansion_coefficients = np.linalg.solve(channel_right_vectors, initial_vector)
edge_vector = model_channel["observables"]["edge"].toarray().reshape(-1, order="F")
readout_factors = edge_vector.conjugate() @ channel_right_vectors
visibility = np.abs(readout_factors * expansion_coefficients)

phase_distance_pi = np.abs(
    np.angle(np.exp(1j * (np.angle(channel_eigenvalues) - np.pi)))
)
pi_candidates = np.where(phase_distance_pi < 0.45)[0]
if len(pi_candidates):
    pi_index = pi_candidates[np.argmax(visibility[pi_candidates])]
else:
    pi_index = int(np.argmin(phase_distance_pi))
lambda_pi = channel_eigenvalues[pi_index]
lambda_steady = channel_eigenvalues[np.argmin(np.abs(channel_eigenvalues - 1.0))]
tau_pi = -p_channel.T / np.log(abs(lambda_pi)) if 0 < abs(lambda_pi) < 1 else np.inf

print("channel construction/diagonalization runtime (s) =", channel_runtime)
print("lambda_steady =", lambda_steady)
print("lambda_pi =", lambda_pi)
print("|lambda_pi| =", abs(lambda_pi))
print("arg(lambda_pi) =", np.angle(lambda_pi))
print("tau_pi / T =", tau_pi / p_channel.T)
print("selected-mode visibility =", visibility[pi_index])
print("spectral radius =", np.max(np.abs(channel_eigenvalues)))

fig, ax = plt.subplots(figsize=(5.5, 5.5))
theta = np.linspace(0, 2 * np.pi, 500)
ax.plot(np.cos(theta), np.sin(theta), "k:", lw=1)
ax.scatter(
    channel_eigenvalues.real,
    channel_eigenvalues.imag,
    s=12,
    alpha=0.55,
    label="channel spectrum",
)
ax.plot(lambda_pi.real, lambda_pi.imag, "ro", ms=8, label=r"visible $\lambda_\pi$")
ax.plot(lambda_steady.real, lambda_steady.imag, "ks", ms=6, label="steady mode")
ax.set(xlabel="Re lambda", ylabel="Im lambda", title="One-period CPTP-map spectrum")
ax.set_aspect("equal", adjustable="box")
ax.legend()
plt.show()


## 12. Production-size local runs

The code below is intentionally disabled in the executed copy. Set RUN_PRODUCTION = True locally after the fast cells pass.

Recommended first production calculation:

- N = 6;
- 80 periods;
- 61 TLS frequencies;
- four samples per drive step;
- validation repeated at representative points;
- checkpoint raw arrays after every completed frequency point.

The expected runtime can exceed five minutes depending on the local BLAS and sparse-matrix implementation.


In [ ]:
def run_production_frequency_sweep():
    p_production = replace(
        p_fast,
        N=6,
        periods=80,
        samples_per_step=4,
        g=0.08,
        gamma1=0.08,
    )
    ratios = np.linspace(0.45, 1.55, 61)
    rows = []
    for index, ratio in enumerate(ratios):
        p = replace(p_production, omega_d=ratio * p_production.Omega / 2.0)
        start = time.perf_counter()
        run = simulate_open(p, validate=False)
        rows.append({"omega_ratio": ratio, **run_metrics(run, discard_periods=20)})
        checkpoint = {
            key: np.asarray([row[key] for row in rows])
            for key in rows[0]
        }
        checkpoint.update({
            "metadata_N": np.asarray(p_production.N),
            "metadata_periods": np.asarray(p_production.periods),
            "metadata_samples_per_step": np.asarray(p_production.samples_per_step),
            "metadata_g": np.asarray(p_production.g),
            "metadata_gamma1": np.asarray(p_production.gamma1),
            "metadata_Omega": np.asarray(p_production.Omega),
            "metadata_T": np.asarray(p_production.T),
            "metadata_discard_periods": np.asarray(20),
        })
        np.savez_compressed("floquet_tls_frequency_production_checkpoint.npz", **checkpoint)
        print(
            f"{index + 1:02d}/{len(ratios)}  "
            f"omega_ratio={ratio:.3f}  "
            f"runtime={time.perf_counter() - start:.1f}s"
        )
    return rows


if RUN_PRODUCTION:
    production_frequency_rows = run_production_frequency_sweep()
else:
    print("Production sweep skipped. Set RUN_PRODUCTION = True on the local machine.")


## 13. Production frequency checkpoint

This section reads the completed \(N=6\), 80-period checkpoint without rerunning the 61-point sweep. The original checkpoint contains observables but not metadata; its parameters are therefore reconstructed explicitly from the production cell above.

A global one-Lorentzian fit is tested only as a falsifiable diagnostic. A separate local fit is restricted to the central emission feature near \(\omega_d=\Omega/2\). Response minima, maxima, and phase singularities are analyzed independently rather than being forced into one line shape.


In [ ]:
checkpoint_name = "floquet_tls_frequency_production_checkpoint.npz"
checkpoint_candidates = [
    Path(checkpoint_name),
    Path("upload") / checkpoint_name,
    Path("numerical_v1") / checkpoint_name,
]
checkpoint_path = next((path for path in checkpoint_candidates if path.exists()), None)
production_available = checkpoint_path is not None

if not production_available:
    print(f"Production checkpoint not found. Place {checkpoint_name} in the working directory.")
else:
    with np.load(checkpoint_path, allow_pickle=False) as checkpoint:
        production_data = {key: np.asarray(checkpoint[key]) for key in checkpoint.files}

    production_observable_keys = [
        "omega_ratio",
        "A_edge",
        "A_tls_transverse",
        "A_tls_population",
        "phase_tls_minus_edge",
        "tls_emission",
        "Mpi_edge_strobe",
        "Mpi_tls_population_strobe",
    ]
    for key in production_observable_keys:
        assert key in production_data, f"missing checkpoint field: {key}"
        assert production_data[key].shape == (61,), f"unexpected shape for {key}"
        assert np.all(np.isfinite(production_data[key])), f"non-finite values in {key}"
    assert np.all(np.diff(production_data["omega_ratio"]) > 0)

    production_metadata = {
        "N": 6,
        "periods": 80,
        "samples_per_step": 4,
        "discard_periods": 20,
        "g": 0.08,
        "gamma1": 0.08,
        "Omega": p_fast.Omega,
        "T": p_fast.T,
        "source": "reconstructed from the production-generation cell",
    }

    x_production = production_data["omega_ratio"]
    edge_production = production_data["A_edge"]
    tls_production = production_data["A_tls_transverse"]
    emission_production = production_data["tls_emission"]

    def fit_lorentzian_window(mask):
        x_fit = x_production[mask]
        y_fit = emission_production[mask]
        initial = [
            float(np.min(y_fit)),
            float(np.max(y_fit) - np.min(y_fit)),
            1.0,
            0.04,
        ]
        parameters, _ = curve_fit(
            lorentzian_with_offset,
            x_fit,
            y_fit,
            p0=initial,
            bounds=([0.0, 0.0, 0.80, 1e-4], [0.02, 0.02, 1.20, 1.0]),
            maxfev=20000,
        )
        prediction = lorentzian_with_offset(x_fit, *parameters)
        residual = np.sum((y_fit - prediction) ** 2)
        total = np.sum((y_fit - np.mean(y_fit)) ** 2)
        r_squared = 1.0 - residual / total
        return parameters, float(r_squared)

    global_fit_parameters, global_fit_r2 = fit_lorentzian_window(
        np.ones_like(x_production, dtype=bool)
    )
    central_fit_mask = (x_production >= 0.945) & (x_production <= 1.055)
    central_fit_parameters, central_fit_r2 = fit_lorentzian_window(central_fit_mask)

    def quadratic_vertex(x_values, y_values, index):
        local = slice(index - 1, index + 2)
        a, b, c = np.polyfit(x_values[local], y_values[local], 2)
        x_vertex = -b / (2.0 * a)
        return float(x_vertex), float(np.polyval([a, b, c], x_vertex))

    def restricted_extremum(values, lower, upper, mode="min"):
        indices = np.where((x_production >= lower) & (x_production <= upper))[0]
        local_index = np.argmin(values[indices]) if mode == "min" else np.argmax(values[indices])
        return int(indices[local_index])

    edge_left_index = restricted_extremum(edge_production, 0.88, 0.98, "min")
    edge_right_index = restricted_extremum(edge_production, 1.02, 1.14, "min")
    tls_left_index = restricted_extremum(tls_production, 0.86, 0.98, "max")
    tls_right_index = restricted_extremum(tls_production, 1.02, 1.14, "max")

    edge_left_dip = quadratic_vertex(x_production, edge_production, edge_left_index)
    edge_right_dip = quadratic_vertex(x_production, edge_production, edge_right_index)
    tls_left_peak = quadratic_vertex(x_production, tls_production, tls_left_index)
    tls_right_peak = quadratic_vertex(x_production, tls_production, tls_right_index)

    edge_split_ratio = edge_right_dip[0] - edge_left_dip[0]
    tls_split_ratio = tls_right_peak[0] - tls_left_peak[0]
    edge_half_split_frequency = 0.5 * edge_split_ratio * p_fast.Omega / 2.0
    tls_half_split_frequency = 0.5 * tls_split_ratio * p_fast.Omega / 2.0

    phase_reliable = (
        (edge_production >= 0.08)
        & (tls_production >= 0.03)
    )
    edge_strobe_correlation = np.corrcoef(
        edge_production, production_data["Mpi_edge_strobe"]
    )[0, 1]

    print("checkpoint =", checkpoint_path)
    print("metadata =", production_metadata)
    print("global Lorentzian [offset, amplitude, center, width] =", global_fit_parameters)
    print("global Lorentzian R^2 =", global_fit_r2)
    print("central Lorentzian [offset, amplitude, center, width] =", central_fit_parameters)
    print("central Lorentzian R^2 =", central_fit_r2)
    print("central resonance center / (Omega/2) =", central_fit_parameters[2])
    print("edge outer dips =", edge_left_dip, edge_right_dip)
    print("TLS transverse peaks =", tls_left_peak, tls_right_peak)
    print("edge half splitting in frequency units =", edge_half_split_frequency)
    print("TLS half splitting in frequency units =", tls_half_split_frequency)
    print("corr(A_edge, |M_pi|) =", edge_strobe_correlation)
    print("phase points excluded near amplitude zeros =", x_production[~phase_reliable])

    fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True)
    axes[0, 0].plot(x_production, edge_production, "o-", ms=4, label=r"continuous $A_e$")
    axes[0, 0].plot(
        x_production, production_data["Mpi_edge_strobe"], "s--", ms=3,
        label=r"stroboscopic $|M_\pi|$",
    )
    for position in [edge_left_dip[0], edge_right_dip[0]]:
        axes[0, 0].axvline(position, color="0.35", ls=":", lw=1)
    axes[0, 0].set(ylabel="edge response", title="Boundary suppression and interference minima")
    axes[0, 0].legend()

    axes[0, 1].plot(x_production, tls_production, "o-", ms=4, color="C1")
    axes[0, 1].plot(*tls_left_peak, "k^", ms=7)
    axes[0, 1].plot(*tls_right_peak, "k^", ms=7, label="quadratic peak estimate")
    axes[0, 1].set(ylabel=r"$|c_d|$", title="Transverse TLS double structure")
    axes[0, 1].legend()

    axes[1, 0].plot(x_production, emission_production, "o", ms=4, label="production data")
    central_dense = np.linspace(0.93, 1.07, 500)
    axes[1, 0].plot(
        central_dense,
        lorentzian_with_offset(central_dense, *central_fit_parameters),
        "-",
        label=fr"local Lorentzian, $R^2={central_fit_r2:.3f}$",
    )
    axes[1, 0].set(
        xlabel=r"$\omega_d/(\Omega/2)$", ylabel=r"$\gamma_1\overline{p_e}$",
        title="Central emission resonance inside a multi-feature spectrum",
    )
    axes[1, 0].legend()

    axes[1, 1].plot(
        x_production, production_data["phase_tls_minus_edge"],
        color="0.75", lw=1, label="wrapped phase",
    )
    axes[1, 1].scatter(
        x_production[phase_reliable],
        production_data["phase_tls_minus_edge"][phase_reliable],
        s=24, label="reliable amplitude",
    )
    axes[1, 1].scatter(
        x_production[~phase_reliable],
        production_data["phase_tls_minus_edge"][~phase_reliable],
        marker="x", color="C3", s=35, label="near an amplitude zero",
    )
    axes[1, 1].set(
        xlabel=r"$\omega_d/(\Omega/2)$", ylabel="relative phase [rad]",
        title="Phase is singular where the edge phasor vanishes",
    )
    axes[1, 1].legend(fontsize=8)
    fig.suptitle("Production frequency sweep: N=6, 80 periods, discard first 20 periods")
    fig.tight_layout()
    plt.show()


## 14. Matched-window control

The fast scan and the production scan change both system size and observation window. To avoid attributing a time-window effect to \(N\), this control repeats the 61-point grid at \(N=4\) while matching the production protocol: 80 periods, four samples per step, and a 20-period discard.

The comparison is normalized only to expose line-shape changes; absolute amplitudes remain available in the arrays and must be used for quantitative claims.


In [ ]:
matched_control_available = production_available and RUN_MATCHED_N4_CONTROL
if not matched_control_available:
    print("Matched-window N=4 control skipped.")
else:
    matched_rows = []
    matched_start = time.perf_counter()
    for ratio in x_production:
        parameters = replace(
            p_fast,
            N=4,
            periods=80,
            samples_per_step=4,
            omega_d=ratio * p_fast.Omega / 2.0,
        )
        run = simulate_open(parameters, validate=False)
        matched_rows.append({
            "omega_ratio": ratio,
            **run_metrics(run, discard_periods=20),
        })
    matched_control_data = {
        key: np.asarray([row[key] for row in matched_rows])
        for key in matched_rows[0]
    }
    matched_runtime = time.perf_counter() - matched_start

    central_window = (x_production >= 0.85) & (x_production <= 1.15)
    central_indices = np.where(central_window)[0]
    matched_tls_peaks = central_indices[
        find_peaks(matched_control_data["A_tls_transverse"][central_window])[0]
    ]
    print("matched-window N=4 runtime (s) =", matched_runtime)
    print(
        "matched-window N=4 TLS local peaks =",
        [
            (x_production[index], matched_control_data["A_tls_transverse"][index])
            for index in matched_tls_peaks
        ],
    )

    def normalize_maximum(values):
        values = np.asarray(values)
        return values / np.max(np.abs(values))

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), sharex=True)
    axes[0].plot(
        frequency_data["omega_ratio"], normalize_maximum(frequency_data["A_edge"]),
        "o--", ms=3, label="N=4, 32 periods, discard 8",
    )
    axes[0].plot(
        x_production, normalize_maximum(matched_control_data["A_edge"]),
        "-", label="N=4, 80 periods, discard 20",
    )
    axes[0].plot(
        x_production, normalize_maximum(edge_production),
        "-", label="N=6, 80 periods, discard 20",
    )
    axes[0].set(
        xlabel=r"$\omega_d/(\Omega/2)$", ylabel="normalized edge amplitude",
        title="Observation-window and size comparison",
    )
    axes[0].legend(fontsize=8)

    axes[1].plot(
        frequency_data["omega_ratio"],
        normalize_maximum(frequency_data["A_tls_transverse"]),
        "o--", ms=3, label="N=4, 32 periods, discard 8",
    )
    axes[1].plot(
        x_production,
        normalize_maximum(matched_control_data["A_tls_transverse"]),
        "-", label="N=4, 80 periods, discard 20",
    )
    axes[1].plot(
        x_production, normalize_maximum(tls_production),
        "-", label="N=6, 80 periods, discard 20",
    )
    axes[1].set(
        xlabel=r"$\omega_d/(\Omega/2)$", ylabel="normalized transverse TLS amplitude",
        title="Single early-time peak evolves into interference structure",
    )
    axes[1].legend(fontsize=8)
    fig.tight_layout()
    plt.show()


## 15. Representative trajectories and window dependence

Five \(N=6\) frequencies are rerun: the two TLS-response maxima, the two deep edge minima, and exact nominal resonance. Each calculation is below the five-minute cell limit. Sliding-window phasors show whether a fixed late-time Fourier amplitude represents a stationary response or samples a beat node.


In [ ]:
representative_available = production_available and RUN_REPRESENTATIVE_N6
if not representative_available:
    print("Representative N=6 trajectories skipped.")
else:
    representative_ratios = [
        x_production[tls_left_index],
        x_production[edge_left_index],
        1.0,
        x_production[edge_right_index],
        x_production[tls_right_index],
    ]
    representative_runs = {}
    representative_start = time.perf_counter()
    for ratio in representative_ratios:
        parameters = replace(
            p_fast,
            N=6,
            periods=80,
            samples_per_step=4,
            omega_d=ratio * p_fast.Omega / 2.0,
        )
        point_start = time.perf_counter()
        run = simulate_open(parameters, validate=False)
        representative_runs[ratio] = run
        print(
            f"ratio={ratio:.6f}, runtime={time.perf_counter() - point_start:.1f}s, "
            f"A_edge(20T)={run_metrics(run, 20)['A_edge']:.5f}, "
            f"A_TLS(20T)={run_metrics(run, 20)['A_tls_transverse']:.5f}"
        )
    print("representative-trajectory total runtime (s) =", time.perf_counter() - representative_start)

    def finite_window_phasor(time_values, signal_values, omega, center_period, width_period, period):
        lower = (center_period - width_period / 2.0) * period
        upper = (center_period + width_period / 2.0) * period
        mask = (time_values >= lower) & (time_values <= upper)
        t_local = time_values[mask]
        signal_local = signal_values[mask]
        signal_local = signal_local - np.mean(signal_local)
        return 2.0 * np.trapezoid(
            signal_local * np.exp(1j * omega * t_local / 2.0), t_local
        ) / (t_local[-1] - t_local[0])

    centers = np.arange(8.0, 73.0, 2.0)
    window_width = 12.0
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
    window_summary = []
    for ratio, run in representative_runs.items():
        periods = np.arange(len(run["stroboscopic"]["edge"]))
        demodulated_edge = ((-1.0) ** periods) * run["stroboscopic"]["edge"]
        axes[0].plot(periods, demodulated_edge, label=f"{ratio:.3f}")

        tls_lowering = 0.5 * (
            run["continuous"]["tls_x"] + 1j * run["continuous"]["tls_y"]
        )
        edge_windows = np.asarray([
            abs(finite_window_phasor(
                run["time"], run["continuous"]["edge"], p_fast.Omega,
                center, window_width, p_fast.T,
            ))
            for center in centers
        ])
        tls_windows = np.asarray([
            abs(finite_window_phasor(
                run["time"], tls_lowering, p_fast.Omega,
                center, window_width, p_fast.T,
            ))
            for center in centers
        ])
        axes[1].plot(centers, edge_windows, label=f"{ratio:.3f}")
        axes[2].plot(centers, tls_windows, label=f"{ratio:.3f}")
        window_summary.append({
            "omega_ratio": ratio,
            "edge_window_min": float(np.min(edge_windows)),
            "edge_window_max": float(np.max(edge_windows)),
            "tls_window_min": float(np.min(tls_windows)),
            "tls_window_max": float(np.max(tls_windows)),
        })

    axes[0].set(
        xlabel="Floquet period n", ylabel=r"$(-1)^n\langle Z_0(nT)\rangle$",
        title="Demodulated edge traces",
    )
    axes[1].set(
        xlabel="window center [periods]", ylabel=r"sliding $A_e$",
        title=f"Edge phasor in {window_width:.0f}T windows",
    )
    axes[2].set(
        xlabel="window center [periods]", ylabel=r"sliding $|c_d|$",
        title=f"TLS phasor in {window_width:.0f}T windows",
    )
    for axis in axes:
        axis.legend(title=r"$\omega_d/(\Omega/2)$", fontsize=7, title_fontsize=8)
    fig.tight_layout()
    plt.show()

    print("sliding-window ranges:")
    for row in window_summary:
        print(row)


## 16. Interpretation after the production run

The production data revise the fast-scan interpretation in three important ways.

1. **The central dissipative resonance survives.** A local Lorentzian describes the emission peak near \(\omega_d=\Omega/2\) well. A single Lorentzian over the complete interval does not: it locks onto a broader low-frequency feature and leaves systematic residuals.
2. **The response amplitude is not a stationary single-pole line shape.** The transverse TLS signal develops two flanking maxima and a central antiresonance, while the edge response develops deep minima. The phase becomes undefined when the edge phasor approaches zero, so those phase points cannot be used in a susceptibility fit.
3. **The structure is strongly observation-window dependent.** The matched \(N=4\) control also develops multiple features, and the representative trajectories show beating in sliding Fourier windows. Therefore the present data are consistent with coherent edge--TLS hybridization and interference, but they do not yet prove two stationary spectral poles or a size-induced splitting.

The next decisive numerical test is a \(g\)-sweep of the dip/peak separation together with channel eigenvalues versus detuning. Linear splitting with the dressed matrix element \(g_m\), accompanied by the corresponding channel-mode avoided crossing, would convert the present interference signature into a quantitative hybridization result.
